# Track 02.3: Production Local Vector Storage with ChromaDB

```
====================================================================================================
RAG ENGINEERING ACADEMY — TRACK 02: EMBEDDINGS & VECTOR SEARCH
Notebook 02.3: Production Local Vector Storage with ChromaDB
Architectural Layer: Storage & Indexing Engine (ChromaDB 1.x)
Execution Environment: Local CPU / SQLite + HNSWlib / Python 3.12 (.venv312)
====================================================================================================
```

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
1. **Understand Vector Database Architecture**: Contrast raw in-memory vector indices (NumPy, standalone HNSW) with production vector databases offering ACID metadata storage, atomic CRUD, and structured querying.
2. **Master ChromaDB Client Lifecycle**: Distinguish between ephemeral in-memory clients (`chromadb.EphemeralClient()`) for unit testing and persistent disk-backed clients (`chromadb.PersistentClient()`) for production deployments.
3. **Configure Distance Metrics**: Mathematically and geometrically compare `cosine`, `l2` (squared Euclidean), and `ip` (inner product) distance spaces in ChromaDB.
4. **Implement Full Document CRUD**: Execute Create (`add`), Read (`get` & `query`), Update (`update`), Upsert (`upsert`), and Delete (`delete`) operations with ID deduplication and schema consistency.
5. **Harness Compound Metadata Filtering**: Execute structured queries combining vector similarity with boolean metadata filters using `$and`, `$or`, `$in`, `$gte`, `$eq`, and `$contains`.
6. **Integrate Embedding Functions**: Ingest pre-computed NumPy vector embeddings directly and integrate custom deterministic `EmbeddingFunction` implementations for 100% offline, zero-dependency execution.

---

## 🏗️ Architectural Overview: ChromaDB Inside-Out

In Track 02.1 and 02.2, we explored vector metric spaces and built HNSW and IVF indices from first principles. However, a production RAG application requires more than just nearest-neighbor calculation:
- **Persistence**: Indices must survive process restarts without rebuilding from scratch.
- **Payload & Metadata Association**: High-dimensional vectors must be coupled with document chunks, source citations, timestamps, and access control lists (ACLs).
- **Hybrid Filter Pruning**: Queries often demand: *"Find the top-5 most relevant paragraphs, BUT only from documents published after 2023 by tenant 'engineering'"*.

ChromaDB solves this by decoupling metadata storage from vector index segments:

```
+-----------------------------------------------------------------------------------------+
|                                    CHROMADB CLIENT                                      |
|            (chromadb.PersistentClient / chromadb.EphemeralClient)                       |
+-----------------------------------------------------------------------------------------+
                                   |                       |
             Metadata & Document Payloads            Vector Proximity Search
                                   v                       v
               +-----------------------+       +------------------------+
               |     SQLite ENGINE     |       |      HNSWlib C++       |
               |  (chroma.sqlite3)     |       |  (Vector Index Segments|
               +-----------------------+       +------------------------+
               | • Collections schema  |       | • Multi-layer graphs   |
               | • Document text       |       | • Dense vector storage |
               | • JSON Metadata fields|       | • Distance calculation |
               | • Inverted term index |       | • Fast ANN top-k       |
               +-----------------------+       +------------------------+
                                   \                      /
                                    \                    /
                                     v                  v
                      +------------------------------------------+
                      |         Compound Filtered Query          |
                      |   (Filtered Candidate Set & Vector Top-K)|
                      +------------------------------------------+
```

In [1]:
# Setup, imports, and system inspection
import os
import shutil
import sqlite3
import tempfile
from pathlib import Path
import numpy as np
import chromadb
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings

print(f"ChromaDB version: {chromadb.__version__}")
print(f"NumPy version:    {np.__version__}")
print("Vector database environment verified successfully.")

ChromaDB version: 1.5.9
NumPy version:    2.5.4
Vector database environment verified successfully.


## 1. Ephemeral vs. Persistent Client Architecture

ChromaDB provides two distinct client modes:

1. **`chromadb.EphemeralClient()` (or `chromadb.Client()`)**:
   - Runs 100% in-memory using an in-memory SQLite database (`:memory:`) and temporary in-memory HNSW graphs.
   - Ideal for unit tests, rapid scratch experiments, and stateless compute jobs where vectors are cached externally.
   - All state is irrevocably lost when the client instance is garbage collected or the Python process terminates.

2. **`chromadb.PersistentClient(path=...)`**:
   - Persists all metadata, collection definitions, and document contents into a local SQLite database (`chroma.sqlite3`).
   - Persists vector graph segments and embeddings as binary files on disk under the specified directory path.
   - Safe for production restart, incremental ingestion, and local desktop/server RAG applications.

Let us instantiate both clients and examine their behaviors.

In [2]:
# 1. Ephemeral Client (In-Memory)
ephemeral_client = chromadb.Client()
ephemeral_col = ephemeral_client.create_collection("temp_scratchpad")
ephemeral_col.add(
    ids=["e_1"],
    embeddings=[[0.1, 0.9]],
    documents=["Ephemeral scratchpad entry"],
    metadatas=[{"env": "test"}]
)
print("Ephemeral collection count:", ephemeral_col.count())

# 2. Persistent Client (Disk-Backed)
persist_dir = Path("./.chroma_demo_storage")
if persist_dir.exists():
    shutil.rmtree(persist_dir)

persistent_client = chromadb.PersistentClient(path=str(persist_dir))
persistent_col = persistent_client.create_collection(
    name="production_kb",
    metadata={"hnsw:space": "cosine"}
)
persistent_col.add(
    ids=["p_1", "p_2"],
    embeddings=[[1.0, 0.0], [0.7071, 0.7071]],
    documents=[
        "ChromaDB persists vector indices and metadata locally.",
        "SQLite manages relational metadata while HNSWlib indexes vectors."
    ],
    metadatas=[{"topic": "storage", "priority": 1}, {"topic": "architecture", "priority": 2}]
)
print("Persistent collection count:", persistent_col.count())
print(f"Persisted to directory: {persist_dir.resolve()}")

Ephemeral collection count: 1
Persistent collection count: 2
Persisted to directory: /Users/bharathm/Documents/Self Learning/RAG_from_scratch/.chroma_demo_storage


### Inspecting ChromaDB's On-Disk Architecture

Let us look inside the persistence directory created by `chromadb.PersistentClient`.
We will observe:
- `chroma.sqlite3`: The relational database containing collection configurations, segment mappings, document texts, and metadata values.
- Segment subdirectories: Contain binary files representing the HNSW graph layers, vector buffers, and segment metadata.

Let us query SQLite directly using Python's built-in `sqlite3` module to inspect how ChromaDB stores our collections and metadata under the hood.

In [3]:
# Inspect disk files and query SQLite schema directly
print("Files in persistence directory:")
for item in sorted(persist_dir.rglob("*")):
    if item.is_file():
        size_kb = item.stat().st_size / 1024
        print(f"  {item.relative_to(persist_dir)} ({size_kb:.2f} KB)")

# Connect directly to chroma.sqlite3 (read-only inspection)
sqlite_path = persist_dir / "chroma.sqlite3"
conn = sqlite3.connect(sqlite_path)
cursor = conn.cursor()

# Query collection definitions
cursor.execute("SELECT id, name, dimension FROM collections;")
collections = cursor.fetchall()
print("\nStored Collections in SQLite:")
for col_id, name, dim in collections:
    print(f"  Collection: '{name}' | UUID: {col_id} | Dimension: {dim}")

# Query stored metadata records
cursor.execute("SELECT id, key, string_value, int_value FROM embedding_metadata LIMIT 5;")
metadata_rows = cursor.fetchall()
print("\nStored Metadata Rows (Sample):")
for mid, key, s_val, i_val in metadata_rows:
    val = s_val if s_val is not None else i_val
    print(f"  Record ID: {mid} | Key: {key} -> {val}")

conn.close()

Files in persistence directory:
  5479baf3-a0b7-4c69-bf51-5cdacb469956/data_level0.bin (14.45 KB)
  5479baf3-a0b7-4c69-bf51-5cdacb469956/header.bin (0.10 KB)
  5479baf3-a0b7-4c69-bf51-5cdacb469956/length.bin (0.39 KB)
  5479baf3-a0b7-4c69-bf51-5cdacb469956/link_lists.bin (0.00 KB)
  chroma.sqlite3 (184.00 KB)

Stored Collections in SQLite:
  Collection: 'production_kb' | UUID: c7123100-7d2c-4e36-ae2d-4d59ef386189 | Dimension: 2

Stored Metadata Rows (Sample):
  Record ID: 1 | Key: chroma:document -> ChromaDB persists vector indices and metadata locally.
  Record ID: 1 | Key: topic -> storage
  Record ID: 1 | Key: priority -> 1
  Record ID: 2 | Key: topic -> architecture
  Record ID: 2 | Key: priority -> 2


## 2. Distance Metric Spaces: `cosine`, `l2`, and `ip`

ChromaDB allows configuring the vector distance metric per collection using the metadata property `{"hnsw:space": "<metric>"}`:

| Metric Space | Formula | Distance to Self ($u=v$) | Orthogonal Vectors ($u \perp v$) | Geometrical Meaning |
|---|---|---|---|---|
| **`cosine`** (Default) | $D_{\text{cosine}}(u,v) = 1 - \frac{u \cdot v}{\|u\|_2 \|v\|_2}$ | $0.0$ | $1.0$ | Angular separation; invariant to vector magnitude. Ideal for text embeddings. |
| **`l2`** | $D_{\text{L2}}(u,v) = \sum_{i} (u_i - v_i)^2 = \|u - v\|_2^2$ | $0.0$ | $\|u\|_2^2 + \|v\|_2^2$ | **Squared Euclidean distance**. Sensitive to magnitude and scale. |
| **`ip`** | $D_{\text{IP}}(u,v) = 1 - (u \cdot v)$ | $0.0$ (if $\|u\|=1$) | $1.0$ | **Inner product distance**. Maximizes dot product when vectors are normalized. |

> **Crucial Implementation Note**: In ChromaDB / HNSWlib, `"l2"` computes the **squared** Euclidean distance $\|u - v\|_2^2$, not the raw square root $\sqrt{\sum (u_i - v_i)^2}$. This avoids expensive `sqrt()` operations during vector graph traversal while preserving exact rank ordering.

Let us verify these three distance metrics empirically with calibrated test vectors.

In [4]:
# Calibrated unit vectors along canonical axes
v_x = [1.0, 0.0]  # Vector along x-axis
v_y = [0.0, 1.0]  # Orthogonal vector along y-axis
v_diag = [0.70710678, 0.70710678]  # 45-degree unit vector

metrics = ["cosine", "l2", "ip"]
client = chromadb.Client()

print(f"{'Space':<10} | {'Query':<6} | {'Target':<6} | {'Reported Distance':<18} | {'Theoretical Value':<18}")
print("-" * 70)

for metric in metrics:
    col = client.create_collection(name=f"metric_{metric}", metadata={"hnsw:space": metric})
    col.add(
        ids=["target_identical", "target_orthogonal", "target_diagonal"],
        embeddings=[v_x, v_y, v_diag],
        documents=["Identical to vx", "Orthogonal to vx", "45 deg to vx"]
    )
    
    # Query with v_x
    results = col.query(query_embeddings=[v_x], n_results=3)
    
    ids = results["ids"][0]
    distances = results["distances"][0]
    
    for doc_id, dist in zip(ids, distances):
        # Calculate theoretical expectation
        if metric == "cosine":
            expected = 0.0 if "identical" in doc_id else (1.0 if "orthogonal" in doc_id else 1.0 - (1.0 * 0.70710678))
        elif metric == "l2":
            expected = 0.0 if "identical" in doc_id else (2.0 if "orthogonal" in doc_id else (1.0 - 0.70710678)**2 + (0.70710678)**2)
        else: # ip
            expected = 0.0 if "identical" in doc_id else (1.0 if "orthogonal" in doc_id else 1.0 - 0.70710678)
            
        print(f"{metric:<10} | {'v_x':<6} | {doc_id:<17} | {dist:<18.5f} | {expected:<18.5f}")

Space      | Query  | Target | Reported Distance  | Theoretical Value 
----------------------------------------------------------------------
cosine     | v_x    | target_identical  | 0.00000            | 0.00000           
cosine     | v_x    | target_diagonal   | 0.29289            | 0.29289           
cosine     | v_x    | target_orthogonal | 1.00000            | 1.00000           
l2         | v_x    | target_identical  | 0.00000            | 0.00000           
l2         | v_x    | target_diagonal   | 0.58579            | 0.58579           
l2         | v_x    | target_orthogonal | 2.00000            | 2.00000           
ip         | v_x    | target_identical  | 0.00000            | 0.00000           
ip         | v_x    | target_diagonal   | 0.29289            | 0.29289           
ip         | v_x    | target_orthogonal | 1.00000            | 1.00000           


## 3. Document CRUD Operations

A production vector store requires full lifecycle document management:
1. **Create (`add`)**: Inserts new items with unique string IDs, document payloads, metadata dictionaries, and embeddings.
2. **Read (`get` & `query`)**:
   - `get()`: Direct key-value retrieval or metadata-filtered retrieval without computing vector distances.
   - `query()`: Nearest-neighbor similarity search ranking candidates by metric distance.
3. **Update (`update`)**: Modifies metadata, document content, or vectors for existing IDs.
4. **Upsert (`upsert`)**: Idempotent insert-or-update. If an ID exists, it is updated; if not, it is created.
5. **Delete (`delete`)**: Removes items by specific ID lists or matching metadata filter criteria.

Let us walk through each operation systematically.

In [5]:
# 1. CREATE: Ingest small curriculum knowledge documents
crud_client = chromadb.Client()
crud_col = crud_client.create_collection("rag_curriculum_crud")

# Ingest initial documents
crud_col.add(
    ids=["chunk_101", "chunk_102", "chunk_103"],
    embeddings=[
        [0.9, 0.1, 0.0],
        [0.1, 0.9, 0.1],
        [0.0, 0.2, 0.9]
    ],
    documents=[
        "Chunking strategies segment long documents into semantically coherent windows.",
        "Dense vector embeddings project natural language tokens into continuous vector spaces.",
        "ChromaDB stores document metadata in SQLite and vector graphs in HNSWlib."
    ],
    metadatas=[
        {"track": 1, "topic": "chunking", "difficulty": "intro"},
        {"track": 2, "topic": "embeddings", "difficulty": "intermediate"},
        {"track": 2, "topic": "vector_db", "difficulty": "intermediate"}
    ]
)
print("Collection size after add:", crud_col.count())

# 2. READ: Direct Key-Value Fetch via get()
direct_fetch = crud_col.get(
    ids=["chunk_101"],
    include=["documents", "metadatas", "embeddings"]
)
print("\n[Direct Fetch via get()]")
print(f"ID:       {direct_fetch['ids'][0]}")
print(f"Document: {direct_fetch['documents'][0]}")
print(f"Metadata: {direct_fetch['metadatas'][0]}")
print(f"Vector:   {direct_fetch['embeddings'][0]}")

# 3. READ: Approximate Nearest Neighbor Search via query()
query_vec = [0.05, 0.15, 0.85]  # Close to chunk_103 (ChromaDB & vector_db)
search_results = crud_col.query(
    query_embeddings=[query_vec],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)
print("\n[Similarity Search via query()]")
for rank, (doc_id, doc, dist) in enumerate(zip(search_results['ids'][0], search_results['documents'][0], search_results['distances'][0]), start=1):
    print(f"  Rank {rank} | ID: {doc_id} | Distance: {dist:.4f} | Content: {doc[:50]}...")

Collection size after add: 3

[Direct Fetch via get()]
ID:       chunk_101
Document: Chunking strategies segment long documents into semantically coherent windows.
Metadata: {'topic': 'chunking', 'difficulty': 'intro', 'track': 1}
Vector:   [0.89999998 0.1        0.        ]

[Similarity Search via query()]
  Rank 1 | ID: chunk_103 | Distance: 0.0075 | Content: ChromaDB stores document metadata in SQLite and ve...
  Rank 2 | ID: chunk_102 | Distance: 1.1275 | Content: Dense vector embeddings project natural language t...


### Update, Upsert, and Idempotency Mechanics

Understanding the difference between `add()`, `update()`, and `upsert()` is critical for data integrity:

- **`add()`**: In ChromaDB 1.x, if an ID already exists in the collection, the duplicate call is non-destructive (the existing document is preserved and not overwritten).
- **`update()`**: Overwrites existing fields for specified IDs. If an ID does not exist, an error is raised.
- **`upsert()`**: Fully idempotent. If the ID exists, it performs an in-place update; if the ID is new, it performs an insert.

> ⚠️ **The Dimensionality Guard Trap**: If you call `update(ids=[...], documents=[...])` without supplying `embeddings`, ChromaDB will invoke the collection's default embedding function (384-dimensional ONNX MiniLM). If your collection was initialized with a different dimension (e.g. 3-dimensional vectors), an `InvalidArgumentError` dimensionality mismatch will occur. Always provide matching embeddings or configure an appropriate collection `embedding_function`!

In [6]:
# 1. Attempting duplicate add (Verify non-destructive idempotency)
crud_col.add(
    ids=["chunk_101"],
    embeddings=[[0.9, 0.1, 0.0]],
    documents=["Attempted duplicate overwrite that should be ignored"],
    metadatas=[{"track": 1, "topic": "chunking", "difficulty": "OVERWRITTEN_BUG"}]
)
check_add = crud_col.get(ids=["chunk_101"])
print("Document after duplicate add:", check_add["documents"][0])
print("Difficulty metadata:", check_add["metadatas"][0]["difficulty"])
assert check_add["metadatas"][0]["difficulty"] == "intro", "Duplicate add must not overwrite!"

# 2. In-place UPDATE (explicitly modifying metadata and document content)
crud_col.update(
    ids=["chunk_101"],
    embeddings=[[0.92, 0.08, 0.0]],
    documents=["Chunking strategies divide documents into coherent windows with overlap."],
    metadatas=[{"track": 1, "topic": "chunking", "difficulty": "advanced", "updated": True}]
)
check_update = crud_col.get(ids=["chunk_101"])
print("\nDocument after explicit update:", check_update["documents"][0])
print("Updated metadata:", check_update["metadatas"][0])

# 3. UPSERT: Update existing chunk_102 and insert brand new chunk_104 simultaneously
crud_col.upsert(
    ids=["chunk_102", "chunk_104"],
    embeddings=[
        [0.12, 0.88, 0.1],  # Updated vector for chunk_102
        [0.5, 0.5, 0.5]     # New vector for chunk_104
    ],
    documents=[
        "Dense vector embeddings map lexical tokens to continuous geometric coordinates.",
        "Evaluation metrics assess RAG precision, recall, and hallucination rates."
    ],
    metadatas=[
        {"track": 2, "topic": "embeddings", "difficulty": "advanced"},
        {"track": 15, "topic": "evaluation", "difficulty": "intro"}
    ]
)
print("\nCollection size after upsert:", crud_col.count())
print("All registered IDs:", sorted(crud_col.get()["ids"]))

Document after duplicate add: Chunking strategies segment long documents into semantically coherent windows.
Difficulty metadata: intro

Document after explicit update: Chunking strategies divide documents into coherent windows with overlap.
Updated metadata: {'track': 1, 'updated': True, 'topic': 'chunking', 'difficulty': 'advanced'}

Collection size after upsert: 4
All registered IDs: ['chunk_101', 'chunk_102', 'chunk_103', 'chunk_104']


### Deleting Documents

ChromaDB supports two deletion patterns:
1. **Explicit ID Deletion**: `col.delete(ids=["chunk_104"])`
2. **Metadata Filter Deletion**: `col.delete(where={"topic": "chunking"})`

Let us execute both patterns and verify collection state.

In [7]:
# 1. Delete by specific ID
crud_col.delete(ids=["chunk_104"])
print("Count after deleting chunk_104:", crud_col.count())
assert "chunk_104" not in crud_col.get()["ids"]

# 2. Delete by metadata filter
crud_col.delete(where={"difficulty": "advanced"})
print("Count after deleting advanced documents:", crud_col.count())
remaining_records = crud_col.get()
print("Remaining IDs:", remaining_records["ids"])
print("Remaining Topics:", [m["topic"] for m in remaining_records["metadatas"]])

Count after deleting chunk_104: 3
Count after deleting advanced documents: 1
Remaining IDs: ['chunk_103']
Remaining Topics: ['vector_db']


## 4. Advanced Metadata Indexing & Boolean Filtering

In production RAG pipelines, pure vector search is almost never used in isolation. Instead, systems use **filtered vector search** to enforce:
- **Tenant Isolation**: `{"tenant_id": {"$eq": "org_482"}}`
- **Document Freshness**: `{"published_year": {"$gte": 2024}}`
- **Source Filtering**: `{"source_type": {"$in": ["pdf", "confluence"]}}`
- **Permission & Security**: `{"access_tier": {"$lte": user_tier}}`

### ChromaDB Query Filter Operators

ChromaDB provides MongoDB-style filter syntax:

| Operator | Type | Description | Example |
|---|---|---|---|
| **`$eq`** | Comparison | Equal to | `{"status": {"$eq": "published"}}` |
| **`$ne`** | Comparison | Not equal to | `{"archived": {"$ne": True}}` |
| **`$gt` / `$gte`** | Comparison | Greater than / or equal | `{"views": {"$gte": 100}}` |
| **`$lt` / `$lte`** | Comparison | Less than / or equal | `{"latency_ms": {"$lt": 50.0}}` |
| **`$in`** | Set | Value is member of set | `{"department": {"$in": ["ai", "security"]}}` |
| **`$nin`** | Set | Value not in set | `{"tag": {"$nin": ["deprecated", "draft"]}}` |
| **`$and`** | Logical | All conditions must hold | `{"$and": [{"year": {"$gte": 2023}}, {"valid": True}]}` |
| **`$or`** | Logical | At least one condition holds | `{"$or": [{"tier": "free"}, {"featured": True}]}` |
| **`$contains`** | Document | Substring match on text | `where_document={"$contains": "retrieval"}` |

Let us populate an enterprise knowledge collection and execute compound queries.

In [8]:
# Populate enterprise test fixture
filter_client = chromadb.Client()
filter_col = filter_client.create_collection("enterprise_knowledge")

filter_col.add(
    ids=["doc_1", "doc_2", "doc_3", "doc_4", "doc_5"],
    embeddings=[
        [1.0, 0.0, 0.0],
        [0.85, 0.15, 0.0],
        [0.1, 0.9, 0.0],
        [0.0, 0.85, 0.15],
        [0.0, 0.1, 0.9]
    ],
    documents=[
        "RAG Architecture Whitepaper: Hybrid BM25 and dense retrieval with cross-encoders.",
        "Q3 Financial Performance Report: Cloud infrastructure expenditure analysis.",
        "Internal Engineering Handbook: Production microservice deployment standards.",
        "Customer Service FAQ: Account recovery and two-factor authentication reset.",
        "AI Governance Policy: Bias detection, red-teaming protocols, and audit logs."
    ],
    metadatas=[
        {"department": "engineering", "year": 2024, "public": True, "confidentiality": 1},
        {"department": "finance", "year": 2023, "public": False, "confidentiality": 3},
        {"department": "engineering", "year": 2023, "public": False, "confidentiality": 2},
        {"department": "support", "year": 2024, "public": True, "confidentiality": 1},
        {"department": "ai_governance", "year": 2024, "public": False, "confidentiality": 3}
    ]
)

print(f"Ingested {filter_col.count()} enterprise documents.")

# Query 1: Compound $and filter (Engineering documents published in 2024 or later)
res_and = filter_col.query(
    query_embeddings=[[1.0, 0.0, 0.0]],
    n_results=5,
    where={
        "$and": [
            {"department": {"$eq": "engineering"}},
            {"year": {"$gte": 2024}}
        ]
    }
)
print("\nQuery 1 ($and: department=engineering AND year>=2024):")
for doc_id, doc in zip(res_and["ids"][0], res_and["documents"][0]):
    print(f"  Matched: [{doc_id}] {doc[:65]}...")

# Query 2: Compound $or filter with $in (Public documents OR confidentiality <= 1)
res_or = filter_col.query(
    query_embeddings=[[0.0, 0.0, 1.0]],
    n_results=5,
    where={
        "$or": [
            {"public": {"$eq": True}},
            {"department": {"$in": ["finance", "ai_governance"]}}
        ]
    }
)
print("\nQuery 2 ($or: public=True OR department IN [finance, ai_governance]):")
for doc_id, meta in zip(res_or["ids"][0], res_or["metadatas"][0]):
    print(f"  Matched: [{doc_id}] Dept: {meta['department']} | Public: {meta['public']}")

# Query 3: Combining where metadata filter WITH where_document content filter
res_hybrid = filter_col.query(
    query_embeddings=[[1.0, 0.0, 0.0]],
    n_results=5,
    where={"confidentiality": {"$lte": 2}},
    where_document={"$contains": "retrieval"}
)
print("\nQuery 3 (Metadata confidentiality<=2 AND Document text contains 'retrieval'):")
for doc_id, doc in zip(res_hybrid["ids"][0], res_hybrid["documents"][0]):
    print(f"  Matched: [{doc_id}] {doc}")

Ingested 5 enterprise documents.

Query 1 ($and: department=engineering AND year>=2024):
  Matched: [doc_1] RAG Architecture Whitepaper: Hybrid BM25 and dense retrieval with...

Query 2 ($or: public=True OR department IN [finance, ai_governance]):
  Matched: [doc_5] Dept: ai_governance | Public: False
  Matched: [doc_4] Dept: support | Public: True
  Matched: [doc_2] Dept: finance | Public: False
  Matched: [doc_1] Dept: engineering | Public: True

Query 3 (Metadata confidentiality<=2 AND Document text contains 'retrieval'):
  Matched: [doc_1] RAG Architecture Whitepaper: Hybrid BM25 and dense retrieval with cross-encoders.


## 5. Embedding Function Integration: Default, Custom & Pre-computed

ChromaDB allows three different patterns for handling vector embeddings:

1. **Default Embedding Function**:
   - Uses Chroma's built-in `all-MiniLM-L6-v2` ONNX model (384 dimensions) running locally on CPU.
   - Converts `documents=["..."]` and `query_texts=["..."]` automatically into dense vectors without external network calls.

2. **Custom `EmbeddingFunction[Documents]`**:
   - Subclass `chromadb.api.types.EmbeddingFunction`.
   - Implement `__call__(self, input: Documents) -> Embeddings`.
   - Enables plugging in any embedding model, API, or offline deterministic projection.

3. **Direct Pre-computed NumPy Ingestion**:
   - Ingest raw vectors using `embeddings=[[...]]`.
   - Completely bypasses text embedding models during ingestion and query.
   - Ideal for production batch pipelines where embeddings are pre-computed on GPU clusters.

Let us implement a deterministic, 100% offline, zero-dependency token hash embedding function to demonstrate the custom embedding interface.

In [9]:
# Implement a custom zero-dependency token hash projection
import zlib

class DeterministicHashEmbeddingFunction(EmbeddingFunction[Documents]):
    """Projects text into an L2-normalized D-dimensional hash space.
    
    100% offline, deterministic, zero-download, and CPU-friendly for educational verification.
    """
    def __init__(self, dim: int = 16):
        self.dim = dim

    def __call__(self, input: Documents) -> Embeddings:
        embeddings = []
        for text in input:
            vec = np.zeros(self.dim, dtype=np.float32)
            # Hash each token deterministically into coordinate space via zlib.crc32
            for word in text.lower().replace(".", "").replace(",", "").split():
                coord = zlib.crc32(word.encode("utf-8")) % self.dim
                vec[coord] += 1.0
            
            # L2 normalize
            norm = np.linalg.norm(vec)
            if norm > 0:
                vec = vec / norm
            embeddings.append(vec.tolist())
        return embeddings

# Instantiate custom embedding function
custom_ef = DeterministicHashEmbeddingFunction(dim=16)

# Register with a new Chroma collection
custom_client = chromadb.Client()
custom_col = custom_client.create_collection(
    name="custom_embedded_kb",
    embedding_function=custom_ef,
    metadata={"hnsw:space": "cosine"}
)

# Ingest plain text documents WITHOUT passing explicit vectors
custom_col.add(
    ids=["article_transformers", "article_astronomy", "article_robotics"],
    documents=[
        "Self-attention mechanisms and transformer neural networks revolutionized natural language processing.",
        "Stellar nucleosynthesis and cosmic microwave background radiation reveal early universe physics.",
        "Robotic kinematics inverse dynamics and feedback control enable autonomous manipulation."
    ],
    metadatas=[
        {"domain": "machine_learning"},
        {"domain": "astrophysics"},
        {"domain": "robotics"}
    ]
)

# Query using query_texts (Chroma invokes custom_ef automatically!)
query_str = "transformer neural attention mechanism"
text_results = custom_col.query(query_texts=[query_str], n_results=2)

print(f"Query Text: '{query_str}'")
for rank, (doc_id, doc, dist) in enumerate(zip(text_results["ids"][0], text_results["documents"][0], text_results["distances"][0]), 1):
    print(f"  Rank {rank} | ID: {doc_id} | Distance: {dist:.4f}")
    print(f"         Snippet: {doc[:75]}...")

assert text_results["ids"][0][0] == "article_transformers", "Custom embedding function should rank AI snippet first!"

Query Text: 'transformer neural attention mechanism'
  Rank 1 | ID: article_transformers | Distance: 0.3876
         Snippet: Self-attention mechanisms and transformer neural networks revolutionized na...
  Rank 2 | ID: article_astronomy | Distance: 0.6838
         Snippet: Stellar nucleosynthesis and cosmic microwave background radiation reveal ea...


### Direct Pre-computed Vector Ingestion

In enterprise RAG systems, embedding generation often occurs upstream (e.g. distributed Ray/Spark workers or dedicated inference microservices).
When pre-computed embeddings are provided directly:
- ChromaDB skips all text-to-vector computation.
- You can store any dimensionality without needing an onnxruntime model in the local client.
- You can query directly using `query_embeddings`.

Let us demonstrate direct pre-computed vector ingestion using NumPy float32 arrays.

In [10]:
# Generate synthetic pre-computed embeddings (e.g. from an upstream embedding cluster)
np.random.seed(42)
num_vectors = 4
dim = 32

# Create normalized vectors
raw_embeddings = np.random.randn(num_vectors, dim).astype(np.float32)
raw_embeddings /= np.linalg.norm(raw_embeddings, axis=1, keepdims=True)

precomputed_client = chromadb.Client()
precomputed_col = precomputed_client.create_collection(
    name="precomputed_pipeline",
    metadata={"hnsw:space": "cosine"}
)

# Ingest precomputed embeddings directly
precomputed_col.add(
    ids=[f"vec_{i}" for i in range(num_vectors)],
    embeddings=raw_embeddings.tolist(),
    documents=[f"Precomputed knowledge record #{i}" for i in range(num_vectors)],
    metadatas=[{"source": "gpu_batch_pipeline", "index": i} for i in range(num_vectors)]
)

print(f"Successfully ingested {precomputed_col.count()} precomputed {dim}-D vectors.")

# Query with precomputed vector (query vector close to vec_2)
query_vector = (raw_embeddings[2] + np.random.randn(dim) * 0.05).astype(np.float32)
query_vector /= np.linalg.norm(query_vector)

query_res = precomputed_col.query(
    query_embeddings=[query_vector.tolist()],
    n_results=2
)

print("\nPrecomputed Query Results:")
for rank, (doc_id, dist) in enumerate(zip(query_res["ids"][0], query_res["distances"][0]), 1):
    print(f"  Rank {rank} | Matched: {doc_id} | Distance: {dist:.5f}")

assert query_res["ids"][0][0] == "vec_2", "Nearest neighbor must be vec_2!"

Successfully ingested 4 precomputed 32-D vectors.

Precomputed Query Results:
  Rank 1 | Matched: vec_2 | Distance: 0.03267
  Rank 2 | Matched: vec_0 | Distance: 0.83695


## 6. Production Engineering Architecture & HNSW Parameter Tuning

When deploying ChromaDB in production, configuring the underlying HNSW index parameters via collection metadata significantly impacts latency and recall:

```
col = client.create_collection(
    name="optimized_production_index",
    metadata={
        "hnsw:space": "cosine",            # Metric: cosine, l2, or ip
        "hnsw:construction_ef": 100,       # Search beam during index construction (trade build time for graph quality)
        "hnsw:M": 16,                      # Maximum bidirectional edges per node (memory vs routing speed)
        "hnsw:search_ef": 50               # Search beam width at query time (latency vs recall tradeoff)
    }
)
```

### Performance & Operational Guidelines

1. **Batch Ingestion Sizing**:
   - Ingest documents in chunks of 500 to 2,000 items per `col.add()` call. Exceeding 10,000 items in a single call increases SQLite transaction locking time and memory spikes.
2. **Metadata Filtering Selectivity**:
   - High-selectivity filters (pruning >90% of candidates) are evaluated by SQLite before or during HNSW exploration. Keep metadata types scalar (strings, integers, floats, booleans) and avoid storing massive JSON blobs in metadata.
3. **Persisted Directory Backups**:
   - In `chromadb.PersistentClient`, backing up the directory requires ensuring no active write transactions are pending. Use SQLite checkpointing or snapshot the directory during quiescent periods.

Let us execute a clean tear-down of our temporary demo storage to verify clean lifecycle cleanup.

In [11]:
# Clean up temporary demo persistence directory created during the lesson
if persist_dir.exists():
    shutil.rmtree(persist_dir)
    print(f"Successfully cleaned up temporary directory: {persist_dir.resolve()}")

print("All ChromaDB operations, CRUD cycles, and metadata queries verified successfully.")

Successfully cleaned up temporary directory: /Users/bharathm/Documents/Self Learning/RAG_from_scratch/.chroma_demo_storage
All ChromaDB operations, CRUD cycles, and metadata queries verified successfully.


---

## 🏁 Track 02.3 Summary & Key Takeaways

1. **Two Client Modes**: `EphemeralClient` provides zero-disk in-memory storage for rapid testing; `PersistentClient` provides rock-solid disk persistence using SQLite and binary HNSW segments.
2. **Distance Spaces**:
   - `"cosine"`: Angular separation ($1 - \cos \theta$), standard for textual embeddings.
   - `"l2"`: Squared Euclidean distance ($\|u - v\|^2$), avoiding square roots.
   - `"ip"`: Inner product distance ($1 - \langle u, v \rangle$).
3. **Full CRUD Support**:
   - `add()` is non-destructive for existing IDs in Chroma 1.x.
   - `update()` modifies records in place (mind the dimensionality guard!).
   - `upsert()` provides idempotent insert-or-update mechanics for continuous pipelines.
   - `delete()` supports both explicit ID lists and dynamic metadata queries.
4. **Compound Filtering**:
   - Combine semantic vector search with boolean predicate logic (`$and`, `$or`, `$in`, `$gte`, `$eq`, `$contains`) to satisfy enterprise tenancy, freshness, and access controls.
5. **Flexible Embedding Integration**:
   - Supports Chroma's local default ONNX model, custom zero-dependency `EmbeddingFunction` implementations, and direct pre-computed NumPy vector ingestion.

---

### 🚀 What's Next: Track 03 — Retrieval Engineering

In Track 02, we mastered dense embeddings, vector indexing (HNSW & IVF), and production storage with ChromaDB.
In **Track 03 (Retrieval Engineering)**, we will combine dense vector search with sparse keyword search:
- **BM25 Lexical Retrieval** from first principles.
- **Hybrid Retrieval & Reciprocal Rank Fusion (RRF)**.
- **Cross-Encoder Reranking** for high-precision second-stage filtering.
- **Contextual Query Transformation**: HyDE, Query Expansion, and Multi-Query Decomposition.